In [27]:
ROOM_TYPE_INT_TO_NAME = {v: k for k, v in {
    "living": 1, "bedroom": 2, "kitchen": 3,
    "bathroom": 4, "balcony": 10, "door": 11, "front_door": 13,
}.items()}

ROOM_COLORS = {
    "living":     "#E07B54",
    "bedroom":    "#F5C96A",
    "kitchen":    "#A8C5A0",
    "bathroom":   "#88B4D4",
    "balcony":    "#C9A8E0",
    "door":       "#888888",
    "front_door": "#444444",
}

def decode_plan(arr, cond):
    coords = arr.T.copy()
    real_mask = cond["src_key_padding_mask"] < 0.5
    room_type_oh = cond["room_types"]
    room_idx_oh  = cond["room_indices"]

    rooms = {}
    for i in np.where(real_mask)[0]:
        rtype_int = int(np.argmax(room_type_oh[i]))
        ridx      = int(np.argmax(room_idx_oh[i]))
        rooms.setdefault((ridx, rtype_int), []).append(coords[i])

    return [(np.array(pts), ROOM_TYPE_INT_TO_NAME.get(rtype_int, str(rtype_int)))
            for (ridx, rtype_int), pts in sorted(rooms.items())]


def render_decoded(rooms, title, ax):
    ax.set_title(title, fontsize=9)
    ax.set_aspect("equal")
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.axhline(0, color="#ccc", lw=0.5); ax.axvline(0, color="#ccc", lw=0.5)
    for pts, name in rooms:
        color = ROOM_COLORS.get(name, "#cccccc")
        if len(pts) >= 3:
            ax.add_patch(plt.Polygon(pts, closed=True, fc=color, ec="black",
                                     alpha=0.6, lw=0.8, label=name))
        ax.scatter(pts[:, 0], pts[:, 1], s=12, c="black", zorder=3)
        if name in ("door", "front_door"):
            cx, cy = pts.mean(axis=0)
            ax.plot(cx, cy, "x", ms=8, mew=2,
                    color="red" if name == "front_door" else "orange", zorder=4)

print("Helpers ready.")


Helpers ready.


In [29]:
import random as _random
import pickle
from floorplan_diffusion.data.dataset import (
    ResPlanDataset, MAX_NUM_POINTS, CORNER_IDX_DIMS, ROOM_IDX_DIMS,
    _normalize_keys, ROOM_TYPE_TO_INT
)

PKL_PATH = PROJECT_ROOT / "data" / "raw" / "ResPlan.pkl"

# Load only the raw plan dicts — much cheaper than building the full dataset
with open(PKL_PATH, "rb") as f:
    raw_plans = pickle.load(f)

DOOR_TYPE_INT       = 11
FRONT_DOOR_TYPE_INT = 13

def plan_has_door_types(plan, *type_ints):
    plan = _normalize_keys(plan.copy())
    present = set()
    for key, tint in ROOM_TYPE_TO_INT.items():
        if plan.get(key) is not None:
            present.add(tint)
    return all(t in present for t in type_ints)

# Find first matching plan and build a tiny single-plan dataset
target_plan = next(
    p for p in raw_plans
    if plan_has_door_types(p, DOOR_TYPE_INT, FRONT_DOOR_TYPE_INT)
)
print(f"Found plan id={target_plan.get('id', 'unknown')}")

# Build a minimal dataset from just that one plan
import tempfile, os
ds = ResPlanDataset.__new__(ResPlanDataset)
ds.set_name = "train"
ds.num_coords = 2
ds.max_num_points = MAX_NUM_POINTS

result = ds._process_plan(target_plan)
if result is None:
    print("Plan was rejected by _process_plan — trying next...")
    target_plan = next(
        p for p in raw_plans
        if plan_has_door_types(p, DOOR_TYPE_INT, FRONT_DOOR_TYPE_INT)
        and ds._process_plan(p) is not None
    )
    result = ds._process_plan(target_plan)

house_array, d_mask, s_mask, g_mask = result
ds.houses     = [house_array]
ds.door_masks = [d_mask]
ds.self_masks = [s_mask]
ds.gen_masks  = [g_mask]

target_idx = 0
print("Dataset ready with 1 plan.")


MemoryError: 

In [ ]:
ROTATION_LABELS = {
    0: "rot=0 (identity)",
    1: "rot=1 (90° CCW)",
    2: "rot=2 (180°)",
    3: "rot=3 (90° CW)",
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3), dpi=80)
fig.suptitle(f"Plan id={target_plan.get('id','?')} — 4 rotations (no flip)", fontsize=9)

for rot in range(4):
    orig_randint = _random.randint
    orig_random  = _random.random
    _random.randint = lambda a, b, r=rot: r
    _random.random  = lambda: 1.0

    arr, cond = ds[0]

    _random.randint = orig_randint
    _random.random  = orig_random

    render_decoded(decode_plan(arr, cond), ROTATION_LABELS[rot], axes[rot])

handles = [plt.Rectangle((0,0),1,1, fc=c, ec="k", lw=0.5) for c in ROOM_COLORS.values()]
fig.legend(handles, list(ROOM_COLORS.keys()), loc="lower center",
           ncol=len(ROOM_COLORS), fontsize=7, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.savefig("rotation_check.png", dpi=80, bbox_inches="tight")
plt.show()
print("Saved to rotation_check.png")


NameError: name 'target_idx' is not defined

Error in callback <function _draw_all_if_interactive at 0x7fa5306679c0> (for post_execute), with arguments args (),kwargs {}:
Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/python3.11/site-packages/IPython/core/events.py", line 100, in trigger
    func(*args, **kwargs)
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/python3.11/site-packages/matplotlib/pyplot.py", line 278, in _draw_all_if_interactive
    draw_all()
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/python3.11/site-packages/matplotlib/_pylab_helpers.py", line 131, in draw_all
    manager.canvas.draw_idle()
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/python3.11/site-packages/matplotlib/backend_bases.py", line 1893, in draw_idle
    self.draw(*args, **kwargs)
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/python3.11/site-packages/matplotlib/backends/backend_agg.py", line 377, in draw
    self.renderer = self.get_renderer()
                    ^^^^^^^^^^^^^^^^^^^
  File "/zhome/70/7/219373/Floorplan-Diffusion-Models/.venv/lib/pyth

<Figure size 2400x600 with 4 Axes>

In [ ]:
print(f"{'Rotation':<12} {'Type':<12} {'Centroid':>20}")
print("-" * 46)

centroids_by_rot = {}
for rot in range(4):
    orig_randint = _random.randint
    orig_random  = _random.random
    _random.randint = lambda a, b: rot
    _random.random  = lambda: 1.0

    arr, cond = ds[target_idx]

    _random.randint = orig_randint
    _random.random  = orig_random

    rooms = decode_plan(arr, cond)
    door_centroids = [(name, pts.mean(axis=0)) for pts, name in rooms
                      if name in ("door", "front_door")]
    centroids_by_rot[rot] = door_centroids
    for name, c in door_centroids:
        print(f"rot={rot:<8}  {name:<12}  ({c[0]:+.4f}, {c[1]:+.4f})")
    print()

print("── Verification: rot=1 centroid should equal (-y0, x0) of rot=0 ──")
for (n0, c0), (n1, c1) in zip(centroids_by_rot[0], centroids_by_rot[1]):
    expected = np.array([-c0[1], c0[0]])
    ok = np.allclose(c1, expected, atol=1e-6)
    print(f"  {n0}: expected ({expected[0]:+.4f},{expected[1]:+.4f})  "
          f"got ({c1[0]:+.4f},{c1[1]:+.4f})  {'✓' if ok else '✗ MISMATCH'}")


Helpers ready.


In [ ]:
print(f"{'Rotation':<12} {'Type':<12} {'Centroid':>20}")
print("-" * 46)

centroids_by_rot = {}
for rot in range(4):
    orig_randint = _random.randint
    orig_random  = _random.random
    _random.randint = lambda a, b: rot
    _random.random  = lambda: 1.0

    arr, cond = ds[target_idx]

    _random.randint = orig_randint
    _random.random  = orig_random

    rooms = decode_plan(arr, cond)
    door_centroids = [(name, pts.mean(axis=0)) for pts, name in rooms
                      if name in ("door", "front_door")]
    centroids_by_rot[rot] = door_centroids
    for name, c in door_centroids:
        print(f"rot={rot:<8}  {name:<12}  ({c[0]:+.4f}, {c[1]:+.4f})")
    print()

print("── Verification: rot=1 centroid should equal (-y0, x0) of rot=0 ──")
for (n0, c0), (n1, c1) in zip(centroids_by_rot[0], centroids_by_rot[1]):
    expected = np.array([-c0[1], c0[0]])
    ok = np.allclose(c1, expected, atol=1e-6)
    print(f"  {n0}: expected ({expected[0]:+.4f},{expected[1]:+.4f})  "
          f"got ({c1[0]:+.4f},{c1[1]:+.4f})  {'✓' if ok else '✗ MISMATCH'}")


MemoryError: 